# CIS I — Assignment 1: Pivot Calibration

This notebook walks through the pivot-calibration scenario from the
assignment (Figure 1): a pointer and a calibration object, each
carrying a marker body, observed by an optical tracker. Your job is to
**design** the marker-sphere layouts and workspace placements, then
**calibrate** the pointer's tip position `p_tip` to within about 2mm.

Everything runs against `src.simulation`, a self-contained simulator that plays the role of the physical tracker +
hand-guided robot + calibration sensor described in the handout. It injects measurement noise — your code never sees ground truth
except where explicitly noted for **validation/reporting** in Step 8 (real
hardware wouldn't give you that; the point there is to report your achieved
accuracy, not to use the ground truth in your calibration math itself).



In [78]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import simulation
from src import vct3, Rot, Frame

from scipy.spatial.transform import Rotation
from scipy.optimize import least_squares


np.set_printoptions(precision=3, suppress=True)


## Step 1: Design the pointer's marker body

Choose approximate local marker-sphere positions `a_ptr,k` (a `List[vct3]`,
relative to the pointer's own coordinate frame) and the pointer tip's
approximate local position `p_tip_approx`. These are your
*nominal design values* — the simulator will independently perturb the
*actual* manufactured positions by a small, noise amount (bounded at
2mm).


In [79]:
ptr_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points, e.g. vct3(x, y, z)
    # vct3(0, -5, 1),
    # vct3(0,  5, 2),
    # vct3(5,  0, 3),
    # vct3(-5, 0, 4),
    vct3(-40, -30, -20),
    vct3(40, -30, 20),
    vct3(40, 30, -20),
    vct3(-35, 35, 25)
]
ptr_tip_approx = vct3(0, 0, -50)  # TODO: vct3, the tip's approximate position in the pointer's local frame


## Step 1.1: Design the calibration object's marker body

Same idea, for the calibration object's local marker positions `a_cal,k`.


In [80]:
cal_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points
    # vct3(0, -2, 1),
    # vct3(0,  3, 2),
    # vct3(5,  0, 3),
    # vct3(-3, 0, 4),
    vct3(-50, -40, -20),
    vct3(50, -40, 20),
    vct3(50, 40, -20),
    vct3(-45, 45, 30)
]


## Step 2: Approximate workspace poses

Specify approximate poses (as `Frame` class from src) for the tracking camera
(`F_tracker`) and the calibration object (`F_cal`), relative to the
workspace. The tracker's usable volume is the tetrahedral region in the
handout's Figure 2 (roughly 100-200cm in front of the tracker, widening
from ~120cm to ~200cm across) — place the calibration object, and later
the pointer, so they're inside that volume.


In [81]:
F_tracker_approx = Frame(Rot(), vct3(250, 250, -1300))  # TODO: Frame, tracker pose relative to the workspace
F_cal_approx = Frame(Rot(), vct3(-100, 300, 300))      # TODO: Frame, calibration object pose relative to the workspace

## Step 3: Build the simulation

Fully provided — this just wires up your Step 1/2 design choices through
`simulation`'s `DefineMarkerBody` / `PlaceMarkerBody` / `CreateTracker`
equivalents. `ptr_placed`/`cal_placed` are `PlacedBody` handles: each pairs
a `MarkerBody` with its hidden actual placement, and gets passed to
`simulation.sample_marker_bodies(...)` whenever you want a reading.


In [82]:
ptr_mb = simulation.define_marker_body("ptr_mb", ptr_local_positions)
cal_mb = simulation.define_marker_body("cal_mb", cal_local_positions)

cal_placed = simulation.place_marker_body(cal_mb, F_cal_approx)
# The pointer's own body doesn't have a meaningful "placement" yet
# It moves every time it's hand-guided (Step 4-6). 
# precision="precise" matches a robot-controlled pose rather than a coarsely, manually placed
# static object (see place_marker_body's docstring).
ptr_placed = simulation.place_marker_body(ptr_mb, Frame.eye(), precision="precise")

ptr = simulation.define_pointer("ptr1", ptr_tip_approx, ptr_mb)
trk = simulation.create_tracker("trk1", F_tracker_approx)

print("simulation built:", ptr_mb.name, cal_mb.name, ptr.name, trk.name)


simulation built: ptr_mb cal_mb ptr1 trk1


## Step 3.1: Refine `a_ptr,k` / `a_cal,k`

> "Observations of marker positions can be used to determine accurate
> estimates of relative positions a_ptr,k of markers on pointer and
> calibration objects... **Hint**: You probably want to take multiple
> readings in order to reduce the uncertainty associated with the a_k
> values."


In [83]:
def refine_marker_body(trk, placed, n_readings=30, passes=3):
    """
    TODO: implement.

    input:
        trk - Tracker to read with
        placed - PlacedBody whose body's nominal marker positions should be
            refined in place (via body.set_marker_positions(...))
        n_readings - int, readings to average per pass
        passes - int, how many times to repeat
    output: None (mutates placed.body's nominal_marker_positions in place)
    """
    body = placed.body
    for _ in range(passes):
        sums = np.zeros((len(body.nominal_marker_positions), 3))
        for _ in range(n_readings):
            obs = simulation.sample_marker_bodies(
                trk, [placed]
            )[body.name]

            local = obs.frame.inv()
            local_points = np.array([
                (local * marker.p).vec.ravel()
                for marker in obs.markers
            ])
            sums += local_points

        mean_points = sums / n_readings
        body.set_marker_positions([vct3(p) for p in mean_points])


refine_marker_body(trk, ptr_placed)
refine_marker_body(trk, cal_placed)


### Validation

In [84]:
def validate_refinement():
    """
    This is a validation function written for `refine_marker_body()`.
    
    Evaluate refinement across 20 noisy trials for each marker layout.
    """
    saved_rng = np.random.get_state()
    seeds = range(20)

    try:
        for name, layout in [
            ("Pointer", ptr_local_positions),
            ("Calibration", cal_local_positions),
        ]:
            results = []

            for seed in seeds:
                np.random.seed(seed)

                body = simulation.define_marker_body(
                    f"validation_{name}",
                    [vct3(p) for p in layout],
                )
                placed = simulation.place_marker_body(
                    body,
                    Frame(Rot.xyz(.2, -.1, .3), vct3(0, 0, 1500)),
                )
                tracker = simulation.create_tracker(
                    "validation_tracker", Frame.eye()
                )


                truth = body.get_actual_marker_positions()
                truth_before = np.array([
                    point.p.vec.ravel().copy() for point in truth
                ])

                def geometry_error():
                    """Return aligned marker RMSE in mm."""
                    F = tracker.procrustes_solver(
                        body.nominal_marker_positions, truth
                    )
                    return np.sqrt(np.mean([
                        (F * estimated - actual.p).norm() ** 2
                        for estimated, actual in zip(
                            body.nominal_marker_positions, truth
                        )
                    ]))

                before = geometry_error()
                refine_marker_body(
                    tracker, placed, n_readings=30, passes=3
                )
                after = geometry_error()

                truth_after = np.array([
                    point.p.vec.ravel()
                    for point in body.get_actual_marker_positions()
                ])
                np.testing.assert_array_equal(
                    truth_before, truth_after
                )
                results.append((before, after))

            results = np.asarray(results)
            assert np.isfinite(results).all(), "Nonfinite geometry error."

            improved = results[:, 1] < results[:, 0]
            print(f"\n{name}: {len(results)} independent noisy trials")
            print(f"Mean RMSE before: {results[:, 0].mean():.4f} mm")
            print(f"Mean RMSE after:  {results[:, 1].mean():.4f} mm")
            print(f"Worst post-refinement RMSE: "
                  f"{results[:, 1].max():.4f} mm")
            print(f"Trials improved: {improved.sum()}/{len(results)}")

            assert results[:, 1].mean() < results[:, 0].mean(), (
                f"{name}: refinement did not improve mean geometry error."
            )
            print("PASS: Mean geometry error decreased; truth unchanged.")

    finally:
        np.random.set_state(saved_rng)


validate_refinement()


Pointer: 20 independent noisy trials
Mean RMSE before: 0.8542 mm
Mean RMSE after:  0.0043 mm
Worst post-refinement RMSE: 0.0065 mm
Trials improved: 20/20
PASS: Mean geometry error decreased; truth unchanged.

Calibration: 20 independent noisy trials
Mean RMSE before: 0.8540 mm
Mean RMSE after:  0.0043 mm
Worst post-refinement RMSE: 0.0065 mm
Trials improved: 20/20
PASS: Mean geometry error decreased; truth unchanged.


## Steps 4-6: Collect pivot-calibration observations

For each observation `j`: hand-guide the pointer near the calibration
object at some desired orientation `R_desired,j`, take one simultaneous
tracker reading of *both* bodies, and read the calibration sensor. Loop
skeleton is provided; you choose how many observations to take and how to
vary `R_desired,j`


In [85]:
N_OBSERVATIONS = 100  # TODO

F_ptr_list = []
F_cal_list = []
sensed_tip_list = []
for j in range(N_OBSERVATIONS):
    R_desired = Rot.xyz(
        *np.random.uniform(-np.pi / 4, np.pi / 4, size=3) # reasoning of the numbers 
    )  # TODO: vary the desired pointer orientation across observations
    hg = simulation.hand_guide_pointer(ptr, ptr_placed, cal_placed, R_desired)
    obs = simulation.sample_marker_bodies(trk, [ptr_placed, cal_placed])
    sensed = simulation.sense_tip(ptr, hg)

    F_ptr_list.append(obs["ptr_mb"].frame)
    F_cal_list.append(obs["cal_mb"].frame)
    sensed_tip_list.append(sensed)

print(f"collected {len(F_ptr_list)} observations")


collected 100 observations


`sensed_tip_list` holds each observation's high-accuracy calibration-sensor
reading (or `None` if that observation ended up outside the sensor's 5mm
range) — useful as an independent sanity check on your calibration later,
since it measures the tip position a completely different way (directly,
rather than through the pointer's marker geometry).


## Step 7: Compute a calibrated value for `p_tip`

This is the assignment's core deliverable — implement it yourself and specify the details of implementation in the report.

**Corner case to consider**: What are the corner cases you want to analyse here?

**Minimum data**: how many observations, and what about their rotations,
are needed for the stacked system to actually be solvable? How do you define whether a system is solvable here? Include that in your report


In [93]:
def compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_tip_list):
    """
    TODO: implement.

    input:
        F_ptr_list - List[Frame], the pointer marker body's pose relative to
            the tracker for each observation j (Fptr,j)
        F_cal_list - List[Frame], the calibration object's pose relative to
            the tracker for each observation j (Fcal,j), same order/length
        sensed_tip_list - List[Optional[vct3]], the calibration sensor's
            reading of the tip position relative to the calibration object
            for each observation j (p_sensed,j), same order/length; an entry
            is None if that observation's tip was out of sensor range
    output:
        p_tip - vct3, the calibrated tip position in the pointer's local frame
        cov_estimate - np.ndarray (3x3), an estimated covariance for p_tip
            (e.g. from the least-squares fit's residuals)
    """
    sensor_readings = sensed_tip_list

    rotations = []
    translations = []
    readings = []

    for F_ptr, F_cal, sensed in zip(
        F_ptr_list, F_cal_list, sensor_readings
    ):
        if sensed is None:
            continue

        s = np.asarray(
            sensed.vec if hasattr(sensed, "vec") else sensed,
            dtype=float,
        ).reshape(-1)

        if np.array_equal(s, [-10000.0, -10000.0, -10000.0]):
            continue

        F_rel = F_cal.inv() * F_ptr
        R_j = F_rel.R.matrix
        t_j = F_rel.p.vec.ravel()

        rotations.append(R_j)
        translations.append(t_j)
        readings.append(s)

    n = len(readings)
    R = np.asarray(rotations)
    t = np.asarray(translations)
    s = np.asarray(readings)
    A = np.vstack([
        np.hstack((R_j, -np.eye(3)))
        for R_j in R
    ])
    b = -t.reshape(-1)

    initial, _, rank, _ = np.linalg.lstsq(A, b, rcond=None)

    if rank < 6:
        raise ValueError("Insufficient pointer rotation diversity.")

    p0 = initial[:3]

    t_X0 = np.mean(
        np.einsum("nij,j->ni", R, p0) + t - s,
        axis=0,
    )

    x0 = np.concatenate((p0, np.zeros(3), t_X0))

    def residual(x):
        """Map nine parameters to stacked position residuals, shape (3*n,)."""
        p = x[:3]
        R_X = Rotation.from_rotvec(x[3:6]).as_matrix()
        t_X = x[6:9]

        tip_from_tracker = np.einsum("nij,j->ni", R, p) + t
        tip_from_sensor = np.einsum("ij,nj->ni", R_X, s) + t_X

        return (tip_from_tracker - tip_from_sensor).reshape(-1)

    result = least_squares(
        residual,
        x0,
        jac="3-point",
        x_scale="jac",
        max_nfev=2000,
    )

    J = result.jac
    _, singular_values, Vt = np.linalg.svd(
        J, full_matrices=False
    )

    threshold = (
        np.finfo(float).eps
        * max(J.shape)
        * singular_values[0]
    )

    if np.count_nonzero(singular_values > threshold) < 9:
        raise ValueError(
            "Joint calibration is rank deficient. Vary both "
            "pointer orientations and sensed tip positions."
        )

    variance = np.sum(result.fun**2) / (3 * n - 9)

    V_scaled = Vt.T / singular_values
    covariance = variance * np.einsum(
        "ik,jk->ij", V_scaled, V_scaled
    )

    p_tip = vct3(result.x[:3])
    cov_estimate = covariance[:3, :3]
    cov_estimate = (cov_estimate + cov_estimate.T) / 2

    return p_tip, cov_estimate

p_tip_est, p_tip_cov = compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_tip_list)


In [94]:
print("calibrated p_tip:", p_tip_est)
print("estimated covariance (mm^2):\n", p_tip_cov)
print("estimated std-dev per axis (mm):", np.sqrt(np.diag(p_tip_cov)))


calibrated p_tip: vct3(x=-0.23, y=-1.75, z=-49.31)
Internal Vector:
[[ -0.229]
 [ -1.752]
 [-49.306]]
estimated covariance (mm^2):
 [[ 0.  0. -0.]
 [ 0.  0. -0.]
 [-0. -0.  0.]]
estimated std-dev per axis (mm): [0.003 0.003 0.003]


### Validation

In [95]:
def validate_calibration():
    """
    This is a validation function written for `compute_pivot_calibration()`.

    Test known-tip recovery across 20 seeds with and without noise.
    """
    for noise_std in [0.0, 0.02]:
        errors = []

        for seed in range(20):
            rng = np.random.default_rng(seed)
            true_tip = vct3(rng.normal([0, 0, -50], 0.5))
            X = Frame(
                Rot.xyz(*rng.uniform(-0.05, 0.05, 3)),
                vct3(rng.uniform(-1, 1, 3)),
            )

            fp, fc, readings = [], [], []

            for _ in range(100):
                R = Rot.xyz(
                    *rng.uniform(-np.pi / 4, np.pi / 4, 3)
                )
                sensed_true = vct3(rng.uniform(-2, 2, 3))

                translation = X * sensed_true - R * true_tip
                translation += vct3(rng.normal(0, noise_std, 3))

                cal = Frame(
                    Rot.xyz(*rng.uniform(-0.3, 0.3, 3)),
                    vct3(rng.uniform(-100, 100, 3)),
                )
                fp.append(cal * Frame(R, translation))
                fc.append(cal)
                readings.append(
                    sensed_true
                    + vct3(rng.normal(0, noise_std / 10, 3))
                )

            estimate, covariance = compute_pivot_calibration(
                fp, fc, readings
            )

            error = (estimate - true_tip).norm()
            assert np.isfinite(error)
            assert covariance.shape == (3, 3)
            assert np.isfinite(covariance).all()
            np.testing.assert_allclose(
                covariance, covariance.T, atol=1e-12
            )
            assert np.linalg.eigvalsh(covariance).min() >= -1e-12
            errors.append(error)

        errors = np.asarray(errors)
        label = "Noise-free" if noise_std == 0 else "Noisy"
        limit = 1e-6 if noise_std == 0 else 0.1

        print(f"\n{label}: {len(errors)} independent trials")
        print(f"Mean tip error: {errors.mean():.6f} mm")
        print(f"Worst tip error: {errors.max():.6f} mm")
        print(
            f"Trials below {limit:g} mm: "
            f"{np.count_nonzero(errors < limit)}/{len(errors)}"
        )

        assert np.all(errors < limit), "Test tolerance exceeded."
        print("PASS: Tip recovery and covariance checks passed.")


validate_calibration()


Noise-free: 20 independent trials
Mean tip error: 0.000000 mm
Worst tip error: 0.000000 mm
Trials below 1e-06 mm: 20/20
PASS: Tip recovery and covariance checks passed.

Noisy: 20 independent trials
Mean tip error: 0.004979 mm
Worst tip error: 0.010105 mm
Trials below 0.1 mm: 20/20
PASS: Tip recovery and covariance checks passed.


## Step 8: Validate against test poses

> "**Hint**: You may want to consider placing additional marker bodies
> close to (but not inside) the cube defined by the test volume. However,
> this will also depend on your design for and placement of the
> calibration object."

The nominal test-pose grid is `theta_x, theta_y, theta_z` each in
`{0, +-pi/6, +-pi/3, +-pi/2}` and `p_x, p_y, p_z` each in
`{300, 300+-100, 300+-200}` mm. For each test pose, the assignment requires a
**single observation only** (no averaging).

Think about why the hint suggests an *additional*, nearby reference body?

In [89]:
near_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points for this reference body
    vct3(-50, -40, -20),
    vct3( 50, -40,  20),
    vct3( 50,  40, -20),
    vct3(-45,  45,  30),
]
F_near_approx = Frame(Rot(), vct3(-100, 300, 300))  # TODO: Frame, placed close to (not inside) the test-pose cube

near_mb = simulation.define_marker_body("near_mb", near_local_positions)
near_placed = simulation.place_marker_body(near_mb, F_near_approx)
refine_marker_body(trk, near_placed)


Refine this new body's geometry the same way you refined the pointer's and
calibration object's (Step 3b) before using it.


Now the single-observation test loop. For each nominal test pose `F_t`:
place the pointer there (small, robot-precision placement error, same as
the real system), take one simultaneous reading of the pointer and your
Step-8 reference body, and predict the tip's position (relative to that
reference body) using your calibrated `p_tip`.

**Ground truth for reporting only**: `PlacedBody.actual_placement` and
`MarkerBody.actual_marker_positions` hold the hidden values the simulator
sampled — you have access to them here only because this notebook doesn't
gate them behind a separate eval mode (per the handout: "During debugging
you will have access to all of the ... values. These will not be available
during evaluation."). Use them *only* to compute and report your achieved
error in this section — using them anywhere in your actual calibration math
(Steps 3b/7) kills the point of the assignment.


In [90]:
import time

thetas = [0, np.pi / 6, -np.pi / 6, np.pi / 3, -np.pi / 3, np.pi / 2, -np.pi / 2]
offsets = [0, 100, -100, 200, -200]
# TODO: the full grid above is large (7^3 * 5^3 = 42,875 combinations) --
# choose a representative subset of (position, orientation) test poses
# rather than every combination.
test_poses = [
    # TODO: build a list of Frame(Rot.xyz(rx, ry, rz), vct3(300+tx, 300+ty, 300+tz))
    # from thetas/offsets above
    Frame(
        Rot.xyz(rx, ry, rz),
        vct3(300 + tx, 300 + ty, 300 + tz),
    )
    for rx in thetas
    for ry in thetas
    for rz in thetas
    for tx in offsets
    for ty in offsets
    for tz in offsets
]

test_results = []  # list of (F_t_nominal, error_mm)

F_refined_to_actual = trk.procrustes_solver(
    near_mb.nominal_marker_positions,
    near_mb.get_actual_marker_positions(),
)

test_start = time.perf_counter()

for F_t_nominal in test_poses:
    # TODO, for each F_t_nominal:
    #   1. test_placed = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise")
    test_placed = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise")
    #   2. obs = simulation.sample_marker_bodies(trk, [test_placed, near_placed])  -- single reading
    obs = simulation.sample_marker_bodies(trk, [test_placed, near_placed])
    #   3. predict the tip position relative to near_placed using p_tip_est
    F_ptr_to_near = obs[near_mb.name].frame.inv() * obs[ptr_mb.name].frame
    predicted_tip = F_ptr_to_near * p_tip_est
    #   4. compute the TRUE tip position relative to near_placed using
    #      test_placed.actual_placement / near_placed.actual_placement (ground truth,
    #      for reporting only -- see markdown above)
    true_tip_world = test_placed.actual_placement * ptr.get_actual_tip_position().p
    true_tip_near_actual = near_placed.actual_placement.inv() * true_tip_world

    #   5. record the error and append (F_t_nominal, error) to test_results
    true_tip_near_refined = F_refined_to_actual.inv() * true_tip_near_actual  # Tommy: align the frame
    error = (predicted_tip - true_tip_near_refined).norm()

    test_results.append((F_t_nominal, error))

test_runtime = time.perf_counter() - test_start


## Achieved performance

Summarize the tip-position error across the test-pose grid against the
handout's 2mm spec.


In [91]:
# TODO: summarize test_results -- e.g. mean/max error, and how many poses
# meet the assignment's ||delta p_t|| <= 2mm spec.
errors = np.array([error for _, error in test_results])
tolerance = 2.0

print(f"Number of test poses: {len(errors)}")
print(f"Mean error: {errors.mean():.3f} mm")
print(f"RMS error: {np.sqrt(np.mean(errors**2)):.3f} mm")
print(f"Maximum error: {errors.max():.3f} mm")
print(f"95th percentile error: {np.percentile(errors, 95):.3f} mm")
print(
    f"Poses with error <= {tolerance:.1f} mm: "
    f"{np.count_nonzero(errors <= tolerance)}/{len(errors)}"
)
print(f"Elapsed time: {test_runtime:.2f} s")

Number of test poses: 42875
Mean error: 0.105 mm
RMS error: 0.122 mm
Maximum error: 0.521 mm
95th percentile error: 0.223 mm
Poses with error <= 2.0 mm: 42875/42875
Elapsed time: 16.37 s


I am really unsure whether this pipeline can work. The test passed though.